# 11 · Wrap-up and take-homes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/11-wrap-up-and-take-homes.ipynb)

*wrap-up · 5 min*

> 🇪🇸 **Cierre y ejercicios para casa** — Qué conecta los bloques 4, 5 y 6, más tres ejercicios para casa.

What connects Blocks 4, 5 and 6, plus three take-home exercises.

## What you will be able to do

- State the one idea that connects the pseudoinverse, deconvolution and Tucker.
- Find the scaling trap in PCA on real, unstandardized data (take-home A).
- Build attention out of two contractions, and mask padded positions (take-home B).
- Compare a CP decomposition against the Tucker one you built (take-home C).

## Setup

Run this first. It installs and imports everything this notebook needs, and nothing else.

> 🇪🇸 Ejecuta esto primero: instala e importa todo lo que este cuaderno necesita.

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer

rng = np.random.default_rng(0)

## What you did today

> 🇪🇸 Lo que hiciste hoy.

1. **Section 01** — learned the vocabulary of tensors (axis, order, shape, slice,
   fiber, unfolding, contraction, decomposition), and that unfolding turns any
   tensor into a matrix without losing anything.
2. **Sections 02 and 05** — argued about what axes *mean*, and found that a batch
   axis and a time axis behave differently even when the shapes look identical.
3. **Sections 03 and 04** — indexed, broadcast, reshaped and transposed real
   tumour data and real medical images, and hit real problems: zero-variance
   pixels, and reshape silently destroying an image.
4. **Sections 06–10** — wrote contractions with `einsum`; solved an unsolvable
   20,433-equation system with the pseudoinverse; used recursion to forecast real
   airline traffic and to find an eigenvector; convolved and deconvolved a real
   photograph; and compressed a real taxi tensor 4.7× with Tucker, which found
   rush hour on its own.

### One idea connects sections 07, 09 and 10

**When a problem has no exact answer or no true inverse, you do not give up —
you find the best stable approximation.** The pseudoinverse does this for linear
systems, Richardson-Lucy for blurred images, and Tucker for tensors that are too
large to keep in full.

> 🇪🇸 Cuando un problema no tiene respuesta exacta ni inversa verdadera, no te
> rindes: buscas la mejor aproximación estable.

## Where to go next

- `torch.einsum` / `tf.einsum` / `jnp.einsum` — **identical syntax** to what you
  used today.
- [`tensorly`](https://tensorly.org) — proper Tucker and CP decompositions.
- `np.linalg` — the rest of Chapter 2: eigendecomposition, `lstsq`, `pinv`, `qr`.
- `scipy.signal` and `skimage.restoration` — convolution and deconvolution
  beyond today.
- The three take-homes below.

### Optional: the same contraction in PyTorch

Everything today was NumPy, because that is what the workshop's real datasets
and verified numbers are built on. The einsum string does not change when you
move to a deep learning framework — only the array type does.

In [ ]:
# Optional. Colab has torch pre-installed; skip this cell if you prefer.
try:
    import torch
    photo = rng.standard_normal((8, 8, 3))
    w = np.array([0.2125, 0.7154, 0.0721])

    np_gray = np.einsum('hwc,c->hw', photo, w)
    pt_gray = torch.einsum('hwc,c->hw', torch.tensor(photo), torch.tensor(w))

    print(np.allclose(np_gray, pt_gray.numpy()))     # True — same string, same answer
except ImportError:
    print("torch not installed — nothing here you need")

---

## Take-home A — How many principal components are enough?

> 🇪🇸 Ejercicio para casa A: ¿cuántas componentes principales bastan?

**Real data contains a trap here. Find it.**

In [ ]:
bc = load_breast_cancer(); X, y = bc.data, bc.target

# TODO 1: Center X, run np.linalg.svd, and compute the fraction of variance each
#         component explains (variance is proportional to S**2).

# TODO 2: How many components explain 95% of the variance? The answer will look
#         TOO GOOD. Do not trust it yet.

# TODO 3: Print X.var(axis=0). The 30 measurements use different units — some are
#         areas in the thousands, some are ratios below 1. What is that doing?

# TODO 4: Redo everything on standardized data: (X - mean) / std. How many now?

# TODO 5: Scatter-plot the first 2 components, coloured by y. Do the two groups
#         separate?

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
Xc = X - X.mean(axis=0)
S = np.linalg.svd(Xc, full_matrices=False)[1]
frac = S**2 / (S**2).sum()
n95 = np.argmax(np.cumsum(frac) >= 0.95) + 1      # 1  (!)
print(n95, round(frac[0], 3))                      # 1 0.982

print(np.sort(X.var(axis=0))[[0, -1]])             # ~0.0000075 up to ~324000

Xs = (X - X.mean(axis=0)) / X.std(axis=0)
S2 = np.linalg.svd(Xs, full_matrices=False)[1]
n95_scaled = np.argmax(np.cumsum(S2**2 / (S2**2).sum()) >= 0.95) + 1   # 10
print(n95_scaled)

# Without standardizing, the first component appears to explain 98.2% of the
# variance. IT IS AN ILLUSION: `worst area` has a variance around 323,000 while
# smoothness values sit below 1, so PCA reports the largest UNIT, not the
# largest PATTERN. After standardizing, the first component explains 44% and
# TEN components are needed.
#
# PCA KNOWS NOTHING ABOUT UNITS. Features on different scales must be
# standardized first.

# TODO 5:
# Z = Xs @ np.linalg.svd(Xs, full_matrices=False)[2][:2].T
# import matplotlib.pyplot as plt
# plt.scatter(Z[:, 0], Z[:, 1], c=y, s=8, cmap="coolwarm")

---

## Take-home B — Attention is two contractions

> 🇪🇸 Ejercicio para casa B: la atención son dos contracciones.

Attention is the mechanism that answers question 5 from section 05: *which parts
of a sequence matter most?* Protein language models use it so every amino acid
can look at every other one; recommenders use it to weight a user's past
interactions.

In [ ]:
np.random.seed(6)
batch, seq_len, dim = 4, 12, 16
Q, K, V = (np.random.randn(batch, seq_len, dim) for _ in range(3))

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x); return e / e.sum(axis=axis, keepdims=True)

# TODO 1: With einsum, compute scores[b,i,j] = how much position i attends to
#         position j. Shape (4, 12, 12). Scale by 1/sqrt(dim).

# TODO 2: Apply softmax on the correct axis so each row of weights sums to 1.

# TODO 3: With einsum, combine V using those weights -> (4, 12, 16).

# TODO 4: Suppose the last 3 positions are padding, not real data. Build a mask,
#         set those scores to -np.inf BEFORE the softmax, and verify the padded
#         positions receive exactly zero weight.

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
scores  = np.einsum('bid,bjd->bij', Q, K) / np.sqrt(dim)
weights = softmax(scores, axis=-1)
output  = np.einsum('bij,bjd->bid', weights, V)
print(scores.shape, weights.shape, output.shape)
print(np.allclose(weights.sum(axis=-1), 1.0))        # True

mask = np.zeros((seq_len, seq_len)); mask[:, -3:] = -np.inf
weights_masked = softmax(scores + mask, axis=-1)
print(weights_masked[..., -3:].max())                # 0.0 — exactly zero weight

# `scores` is Chapter 2's dot product (eq. 2.8); `output` is Chapter 2's linear
# combination (eq. 2.28). ATTENTION IS TWO CONTRACTIONS built from ideas you had
# already read.
#
# TODO 4 solves the variable-length problem from section 02: THE MASK IS HOW
# REAL MODELS HANDLE SEQUENCES AND VIDEOS OF DIFFERENT LENGTHS.

---

## Take-home C — CP decomposition, compared to Tucker

> 🇪🇸 Ejercicio para casa C: CP comparado con Tucker.

In [ ]:
# TODO 1: Build one rank-1 tensor with einsum from three random vectors of
#         length 4, 5 and 24. What shape is it? How many numbers define it?

# TODO 2: Compare that against 4*5*24. What is the compression of ONE rank-1 piece?

# TODO 3: pip install tensorly, run tensorly.decomposition.parafac on the taxi
#         tensor T with rank=3, and compare its error against your Tucker result.

# TODO 4: Which was more accurate at similar size? Why might that be?

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
a, b, c = rng.standard_normal(4), rng.standard_normal(5), rng.standard_normal(24)
rank1 = np.einsum('i,j,k->ijk', a, b, c)     # (4, 5, 24) from only 33 numbers
print(rank1.shape, len(a) + len(b) + len(c), 4 * 5 * 24)   # (4,5,24) 33 480
print(round(480 / 33, 1))                                   # 14.5x for one piece

# TODO 3 — needs the taxi tensor from section 10:
# %pip install -q tensorly
# import tensorly as tl
# from tensorly.decomposition import parafac
# cp = parafac(tl.tensor(T), rank=3)
# err = tl.norm(tl.cp_to_tensor(cp) - T) / tl.norm(T)

# TUCKER IS USUALLY MORE ACCURATE AT EQUAL SIZE, because its dense core can
# represent interactions between components on different axes — something CP's
# strict sum of rank-1 pieces cannot do. CP is often preferred when
# interpretability matters, because each component is one simple pattern per
# axis.

## Thank you

> 🇪🇸 Gracias por venir. Pregunta en Discord en español o en inglés — lo que te
> permita preguntar más rápido.

Questions stay welcome in Discord, in Spanish or English. The
[handbook](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)
has everything from today, including the facilitator notes.

---

## Done with this section

That is the whole workshop. Thank you for coming.

[← Back to the workshop site](https://project-delphi.github.io/tensors-workshop/) · [All notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)